## Importación de librerías necesarias para el procesamiento de datos

In [ ]:
# ============================================================
# IMPORTACIÓN DE LIBRERÍAS
# ============================================================

import os
import re
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Conv1D,
    MaxPooling1D,
    LSTM,
    Dense,
    Dropout
)



import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    roc_auc_score
)
# Importar librerías para visualización de datos y métricas de evaluación
import seaborn as sns                    # Visualizaciones estadísticas elegantes
from time import time

In [ ]:
# ============================================================
# REPRODUCIBILIDAD DEL EXPERIMENTO
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

print("✅ Semilla fijada:", SEED)

✅ Semilla fijada: 42


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Carga de Datasets

In [ ]:
# ============================================================
# PARSER OTIDS V2
# Compatible con registros DLC = 0 sin payload
# ============================================================

import os
import re
import gc
import pandas as pd

folder = "/content/drive/MyDrive/Datasets"

files = [
    "Attack_free_dataset.txt",
    "DoS_attack_dataset.txt",
    "Fuzzy_attack_dataset.txt",
    "Impersonation_attack_dataset.txt"
]

# Verificar archivos
print("Archivos encontrados:")

for file_name in files:
    file_path = os.path.join(folder, file_name)

    if not os.path.isfile(file_path):
        raise FileNotFoundError(
            f"No se encontró: {file_path}"
        )

    print("✅", file_name)


# ============================================================
# REGEX V2
# Payload opcional cuando DLC = 0
# ============================================================

pattern = re.compile(
    r"Timestamp:\s+([\d.]+)\s+"
    r"ID:\s+([0-9A-Fa-f]+\s+[0-9]{3})\s+"
    r"DLC:\s+(\d+)"
    r"(?:\s+(.*))?$"
)


# ============================================================
# EXTRAER REGISTROS
# ============================================================

all_data = []

print("\n================ CARGA DE DATASETS ================")

for file_name in files:

    file_path = os.path.join(folder, file_name)

    count = 0

    print(f"\nLeyendo: {file_name}")

    with open(
        file_path,
        "r",
        encoding="latin-1",
        errors="ignore"
    ) as f:

        for line in f:

            match = pattern.search(line.strip())

            if match:

                timestamp = match.group(1)
                can_id = match.group(2)
                dlc = match.group(3)
                data = match.group(4)

                if data is None:
                    data = ""

                all_data.append({
                    "file": file_name,
                    "timestamp": float(timestamp),
                    "id": can_id,
                    "dlc": int(dlc),
                    "data": data.strip()
                })

                count += 1

    print(f"Registros extraídos: {count:,}")


# ============================================================
# DATAFRAME
# ============================================================

df_all = pd.DataFrame(
    all_data,
    columns=[
        "file",
        "timestamp",
        "id",
        "dlc",
        "data"
    ]
)

del all_data
gc.collect()


print("\n================ RESULTADO PARSER V2 ================")

print("Dimensión df_all:", df_all.shape)

print("\nRegistros por archivo:")
print(df_all["file"].value_counts())

print("\nRegistros DLC = 0:")
print(f"{(df_all['dlc'] == 0).sum():,}")

Archivos encontrados:
✅ Attack_free_dataset.txt
✅ DoS_attack_dataset.txt
✅ Fuzzy_attack_dataset.txt
✅ Impersonation_attack_dataset.txt

================ CARGA DE DATASETS ================

Leyendo: Attack_free_dataset.txt
Registros extraídos: 2,369,398

Leyendo: DoS_attack_dataset.txt
Registros extraídos: 656,579

Leyendo: Fuzzy_attack_dataset.txt
Registros extraídos: 591,990

Leyendo: Impersonation_attack_dataset.txt
Registros extraídos: 995,472

================ RESULTADO PARSER V2 ================
Dimensión df_all: (4613439, 5)

Registros por archivo:
file
Attack_free_dataset.txt             2369398
Impersonation_attack_dataset.txt     995472
DoS_attack_dataset.txt               656579
Fuzzy_attack_dataset.txt             591990
Name: count, dtype: int64

Registros DLC = 0:
123,754


In [ ]:
# ============================================================
# VALIDACIÓN CAN V2
# Compatible con DLC = 0
# ============================================================

import re
import pandas as pd


def is_valid_can_record(row):
    """
    Validación conjunta de DLC y payload.

    Reglas:
    - DLC debe estar entre 0 y 8.
    - DLC = 0 implica payload vacío válido.
    - DLC > 0 requiere exactamente DLC bytes.
    - Cada byte debe ser hexadecimal válido (00-FF).
    """

    # --------------------------------------------------------
    # DLC
    # --------------------------------------------------------
    try:
        dlc = int(row["dlc"])
    except (ValueError, TypeError):
        return False

    if not 0 <= dlc <= 8:
        return False

    # --------------------------------------------------------
    # Payload
    # --------------------------------------------------------
    data = row["data"]

    if pd.isna(data):
        data_str = ""
    else:
        data_str = str(data).strip()

    # --------------------------------------------------------
    # DLC = 0
    # Payload vacío es válido
    # --------------------------------------------------------
    if dlc == 0:
        return data_str == ""

    # --------------------------------------------------------
    # DLC > 0
    # Debe existir payload
    # --------------------------------------------------------
    if data_str == "":
        return False

    parts = data_str.split()

    # Número de bytes debe coincidir con DLC
    if len(parts) != dlc:
        return False

    # --------------------------------------------------------
    # Validación hexadecimal byte a byte
    # --------------------------------------------------------
    for part in parts:

        if re.fullmatch(
            r"[0-9A-Fa-f]{1,2}",
            part
        ) is None:
            return False

        try:
            value = int(part, 16)
        except ValueError:
            return False

        if not 0 <= value <= 255:
            return False

    return True


# ============================================================
# APLICAR VALIDACIÓN
# ============================================================

print(
    "Registros antes de validación:",
    f"{len(df_all):,}"
)

valid_can_mask = df_all.apply(
    is_valid_can_record,
    axis=1
)

invalid_can = (
    df_all.loc[~valid_can_mask]
    .copy()
)

df_clean = (
    df_all.loc[valid_can_mask]
    .copy()
    .reset_index(drop=True)
)


# ============================================================
# RESULTADOS
# ============================================================

print("\n==============================================")
print("VALIDACIÓN CAN V2")
print("==============================================")

print(
    "Registros válidos:",
    f"{len(df_clean):,}"
)

print(
    "Registros inválidos:",
    f"{len(invalid_can):,}"
)

print("\nInválidos por archivo:")

if len(invalid_can) > 0:
    print(
        invalid_can["file"].value_counts()
    )
else:
    print("Ninguno")


print("\nDistribución final:")
print(
    df_clean["file"].value_counts()
)


# ============================================================
# DLC = 0
# ============================================================

print("\n==============================================")
print("COMPROBACIÓN DLC = 0")
print("==============================================")

print(
    "DLC = 0 antes:",
    f"{(df_all['dlc'] == 0).sum():,}"
)

print(
    "DLC = 0 después:",
    f"{(df_clean['dlc'] == 0).sum():,}"
)


# ============================================================
# ASSERTS DE SEGURIDAD
# ============================================================

assert len(df_all) == 4613439
assert len(df_clean) == 4613437
assert len(invalid_can) == 2

assert (
    (df_clean["dlc"] == 0).sum()
    == 123754
)

print(
    "\n✅ df_clean V2 reconstruido correctamente."
)

Registros antes de validación: 4,613,439

VALIDACIÓN CAN V2
Registros válidos: 4,613,437
Registros inválidos: 2

Inválidos por archivo:
file
Fuzzy_attack_dataset.txt    2
Name: count, dtype: int64

Distribución final:
file
Attack_free_dataset.txt             2369398
Impersonation_attack_dataset.txt     995472
DoS_attack_dataset.txt               656579
Fuzzy_attack_dataset.txt             591988
Name: count, dtype: int64

COMPROBACIÓN DLC = 0
DLC = 0 antes: 123,754
DLC = 0 después: 123,754

✅ df_clean V2 reconstruido correctamente.


In [ ]:
# ============================================================
# EXPANSIÓN DEL PAYLOAD CAN A 8 BYTES - V2
# Compatible con DLC = 0
# ============================================================

import numpy as np


def expand_data(data_str):
    """
    Convierte el payload hexadecimal CAN a una representación
    decimal de longitud fija igual a 8.

    DLC = 0 -> [0,0,0,0,0,0,0,0]
    DLC < 8 -> zero-padding
    DLC = 8 -> conserva los 8 bytes
    """

    data_str = str(data_str).strip()

    # DLC = 0 / payload vacío
    if data_str == "":
        values = []

    else:
        values = [
            int(part, 16)
            for part in data_str.split()
        ]

    # Zero-padding hasta 8 bytes
    if len(values) < 8:
        values.extend(
            [0] * (8 - len(values))
        )

    return values[:8]


# ============================================================
# APLICAR EXCLUSIVAMENTE SOBRE df_clean
# ============================================================

data_expanded = df_clean["data"].apply(
    expand_data
)

data_array = np.stack(
    data_expanded.to_numpy()
).astype(np.float32)


# ============================================================
# VERIFICACIONES
# ============================================================

print("df_clean:", df_clean.shape)
print("data_array:", data_array.shape)

assert data_array.shape[0] == len(df_clean)
assert data_array.shape[1] == 8


# ============================================================
# COMPROBAR DLC = 0
# ============================================================

mask_dlc0 = (
    df_clean["dlc"].to_numpy() == 0
)

print(
    "\nRegistros DLC = 0:",
    f"{mask_dlc0.sum():,}"
)

print(
    "Primeros payloads expandidos con DLC = 0:"
)

print(
    data_array[mask_dlc0][:5]
)

# Todos los DLC=0 deben tener ocho ceros
assert np.all(
    data_array[mask_dlc0] == 0
)

print(
    "\n✅ Payloads correctamente expandidos."
)
print(
    "✅ Los 123,754 registros DLC = 0 se representan con 8 ceros."
)

df_clean: (4613437, 5)
data_array: (4613437, 8)

Registros DLC = 0: 123,754
Primeros payloads expandidos con DLC = 0:
[[0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0.]]

✅ Payloads correctamente expandidos.
✅ Los 123,754 registros DLC = 0 se representan con 8 ceros.


In [ ]:
# ============================================================
# CONVERSIÓN DEL CAN ID - V2
# ============================================================

import numpy as np

def convert_can_id(value):

    try:
        first_value = str(value).strip().split()[0]
        return int(first_value, 16)

    except Exception:
        return np.nan


df_clean["id"] = (
    df_clean["id"]
    .apply(convert_can_id)
)

df_clean["dlc"] = (
    df_clean["dlc"]
    .astype(int)
)

print("Valores nulos en CAN ID:", df_clean["id"].isna().sum())

assert df_clean["id"].isna().sum() == 0

print("\nPrimeros CAN ID convertidos:")
print(
    df_clean[
        ["timestamp", "id", "dlc", "data"]
    ].head()
)

print("\n✅ CAN ID convertido correctamente.")

Valores nulos en CAN ID: 0

Primeros CAN ID convertidos:
   timestamp   id  dlc                     data
0   0.000000  790    8  05 20 ea 0a 20 1a 00 7f
1   0.000224  809    8  d7 a7 7f 8c 11 2f 00 10
2   0.000462  128    8  00 17 ea 0a 20 1a 20 43
3   0.000704  129    8  7f 84 60 00 00 00 00 53
4   0.000878  288    4              00 00 00 00

✅ CAN ID convertido correctamente.


In [ ]:
# ============================================================
# PREPARACIÓN DE CARACTERÍSTICAS - V2
# ============================================================

# Timestamp SOLO para orden y trazabilidad.
# NO se utiliza como predictor.
timestamps = df_clean[
    "timestamp"
].to_numpy()

# CAN ID + DLC + DATA0...DATA7
features_raw = np.hstack([
    df_clean[
        ["id", "dlc"]
    ].to_numpy(dtype=np.float32),

    data_array
]).astype(np.float32)


print("timestamps:", timestamps.shape)
print("features_raw:", features_raw.shape)

assert features_raw.shape[0] == len(df_clean)
assert features_raw.shape[1] == 10

print(
    "\n✅ Cada mensaje CAN contiene "
    "10 características predictivas."
)

timestamps: (4613437,)
features_raw: (4613437, 10)

✅ Cada mensaje CAN contiene 10 características predictivas.


In [ ]:
# ============================================================
# CODIFICACIÓN DE ETIQUETAS - V2
# ============================================================

from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

labels_message = label_encoder.fit_transform(
    df_clean["file"]
)

class_names = label_encoder.classes_

print("Clases:")

for class_id, class_name in enumerate(class_names):
    print(
        class_id,
        "->",
        class_name
    )

print(
    "\nlabels_message:",
    labels_message.shape
)

assert len(labels_message) == len(df_clean)
assert len(class_names) == 4

print(
    "\n✅ Etiquetas de escenario correctamente codificadas."
)

Clases:
0 -> Attack_free_dataset.txt
1 -> DoS_attack_dataset.txt
2 -> Fuzzy_attack_dataset.txt
3 -> Impersonation_attack_dataset.txt

labels_message: (4613437,)

✅ Etiquetas de escenario correctamente codificadas.


In [ ]:
# ============================================================
# VERIFICACIÓN DE ALINEAMIENTO
# ============================================================

assert (
    features_raw.shape[0]
    == len(timestamps)
    == len(labels_message)
    == len(df_clean)
)

print(
    "✅ Features, timestamps and labels "
    "are correctly aligned."
)

✅ Features, timestamps and labels are correctly aligned.


In [ ]:
# ============================================================
# CONSTRUCCIÓN DE VENTANAS NO SOLAPADAS - V2
# ============================================================

WINDOW_SIZE = 20

X_list = []
y_list = []
ts_list = []

file_names = df_clean[
    "file"
].to_numpy()


print(
    "=============================================="
)
print(
    "SEQUENTIAL WINDOW CONSTRUCTION - V2"
)
print(
    "=============================================="
)


for file_name in class_names:

    mask = (
        file_names == file_name
    )

    features_file = features_raw[mask]
    timestamps_file = timestamps[mask]
    labels_file = labels_message[mask]

    # Número de ventanas completas
    n_windows = (
        len(features_file)
        // WINDOW_SIZE
    )

    n_used = (
        n_windows
        * WINDOW_SIZE
    )

    # Solo mensajes que forman ventanas completas
    features_file = (
        features_file[:n_used]
    )

    timestamps_file = (
        timestamps_file[:n_used]
    )

    labels_file = (
        labels_file[:n_used]
    )

    # 20 mensajes × 10 características
    X_file = features_file.reshape(
        n_windows,
        WINDOW_SIZE,
        10
    )

    # Etiqueta central de la ventana
    # Como cada ventana pertenece a un solo archivo,
    # toda la ventana comparte el mismo escenario.
    y_file = labels_file[
        WINDOW_SIZE // 2:
        n_used:
        WINDOW_SIZE
    ]

    # Timestamp inicial de la ventana
    ts_file = timestamps_file[
        0:
        n_used:
        WINDOW_SIZE
    ]

    X_list.append(X_file)
    y_list.append(y_file)
    ts_list.append(ts_file)

    remainder = (
        len(features_raw[mask])
        - n_used
    )

    print(
        f"{file_name}: "
        f"{n_windows:,} ventanas | "
        f"{remainder} mensajes residuales"
    )


X = np.concatenate(
    X_list,
    axis=0
).astype(np.float32)

y = np.concatenate(
    y_list,
    axis=0
).astype(np.int32)

ts = np.concatenate(
    ts_list,
    axis=0
)


print("\n==============================================")
print("DATASET SECUENCIAL FINAL")
print("==============================================")

print("X:", X.shape)
print("y:", y.shape)
print("ts:", ts.shape)


assert X.shape == (
    230669,
    20,
    10
)

assert y.shape == (
    230669,
)

assert ts.shape == (
    230669,
)

print(
    "\n✅ Dataset secuencial V2 correctamente construido."
)

SEQUENTIAL WINDOW CONSTRUCTION - V2
Attack_free_dataset.txt: 118,469 ventanas | 18 mensajes residuales
DoS_attack_dataset.txt: 32,828 ventanas | 19 mensajes residuales
Fuzzy_attack_dataset.txt: 29,599 ventanas | 8 mensajes residuales
Impersonation_attack_dataset.txt: 49,773 ventanas | 12 mensajes residuales

DATASET SECUENCIAL FINAL
X: (230669, 20, 10)
y: (230669,)
ts: (230669,)

✅ Dataset secuencial V2 correctamente construido.


In [ ]:
# ============================================================
# DIVISIÓN TRAIN / VALIDATION / TEST - V2
# 64% Training - 16% Validation - 20% Held-out Test
# ============================================================

from sklearn.model_selection import train_test_split
import numpy as np

SEED = 42

# ------------------------------------------------------------
# 1. Reservar 20% para Held-out Test
# ------------------------------------------------------------

X_dev, X_test, y_dev, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=SEED
)

# ------------------------------------------------------------
# 2. Del 80% restante, 20% para Validation
#    0.80 × 0.20 = 0.16 del total
# ------------------------------------------------------------

X_train, X_val, y_train, y_val = train_test_split(
    X_dev,
    y_dev,
    test_size=0.20,
    stratify=y_dev,
    random_state=SEED
)

# ------------------------------------------------------------
# VERIFICACIONES
# ------------------------------------------------------------

print("==============================================")
print("TRAIN / VALIDATION / TEST - V2")
print("==============================================")

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Held-out Test:", X_test.shape)

total = (
    len(X_train)
    + len(X_val)
    + len(X_test)
)

print("\nTotal:", total)

assert total == len(X)

assert X_train.shape[1:] == (20, 10)
assert X_val.shape[1:] == (20, 10)
assert X_test.shape[1:] == (20, 10)

print("\n✅ Split V2 correctamente realizado.")

TRAIN / VALIDATION / TEST - V2
Training: (147628, 20, 10)
Validation: (36907, 20, 10)
Held-out Test: (46134, 20, 10)

Total: 230669

✅ Split V2 correctamente realizado.


In [ ]:
# ============================================================
# VERIFICACIÓN DE DISTRIBUCIÓN DE CLASES
# ============================================================

def show_class_distribution(name, labels):

    unique, counts = np.unique(
        labels,
        return_counts=True
    )

    print(f"\n{name}")

    for class_id, count in zip(unique, counts):

        percentage = (
            count / len(labels) * 100
        )

        print(
            f"{class_names[class_id]}: "
            f"{count:,} "
            f"({percentage:.4f}%)"
        )


show_class_distribution(
    "TRAINING",
    y_train
)

show_class_distribution(
    "VALIDATION",
    y_val
)

show_class_distribution(
    "HELD-OUT TEST",
    y_test
)


TRAINING
Attack_free_dataset.txt: 75,820 (51.3588%)
DoS_attack_dataset.txt: 21,010 (14.2317%)
Fuzzy_attack_dataset.txt: 18,943 (12.8316%)
Impersonation_attack_dataset.txt: 31,855 (21.5779%)

VALIDATION
Attack_free_dataset.txt: 18,955 (51.3588%)
DoS_attack_dataset.txt: 5,253 (14.2331%)
Fuzzy_attack_dataset.txt: 4,736 (12.8323%)
Impersonation_attack_dataset.txt: 7,963 (21.5759%)

HELD-OUT TEST
Attack_free_dataset.txt: 23,694 (51.3591%)
DoS_attack_dataset.txt: 6,565 (14.2303%)
Fuzzy_attack_dataset.txt: 5,920 (12.8322%)
Impersonation_attack_dataset.txt: 9,955 (21.5784%)


In [ ]:
# ============================================================
# NORMALIZACIÓN MIN-MAX - V2
# FIT EXCLUSIVAMENTE SOBRE TRAINING
# ============================================================

from sklearn.preprocessing import MinMaxScaler
import numpy as np

scaler = MinMaxScaler()

N_FEATURES = X_train.shape[2]

# ------------------------------------------------------------
# 1. Reshape Training
# ------------------------------------------------------------

X_train_2d = X_train.reshape(
    -1,
    N_FEATURES
)

# ------------------------------------------------------------
# 2. Fit SOLO con Training
# ------------------------------------------------------------

scaler.fit(
    X_train_2d
)

# ------------------------------------------------------------
# 3. Transformar Training
# ------------------------------------------------------------

X_train = scaler.transform(
    X_train.reshape(-1, N_FEATURES)
).reshape(
    X_train.shape
).astype(np.float32)

# ------------------------------------------------------------
# 4. Transformar Validation
# SIN volver a ajustar scaler
# ------------------------------------------------------------

X_val = scaler.transform(
    X_val.reshape(-1, N_FEATURES)
).reshape(
    X_val.shape
).astype(np.float32)

# ------------------------------------------------------------
# 5. Transformar Held-out Test
# SIN volver a ajustar scaler
# ------------------------------------------------------------

X_test = scaler.transform(
    X_test.reshape(-1, N_FEATURES)
).reshape(
    X_test.shape
).astype(np.float32)


# ============================================================
# VERIFICACIONES
# ============================================================

print("==============================================")
print("NORMALIZACIÓN V2")
print("==============================================")

print(
    "Training normalizado:",
    X_train.shape
)

print(
    "Validation normalizado:",
    X_val.shape
)

print(
    "Held-out Test normalizado:",
    X_test.shape
)

assert X_train.shape == (
    147628,
    20,
    10
)

assert X_val.shape == (
    36907,
    20,
    10
)

assert X_test.shape == (
    46134,
    20,
    10
)

print(
    "\n✅ MinMaxScaler ajustado "
    "exclusivamente con Training."
)

print(
    "✅ Validation y Test fueron "
    "solo transformados."
)

NORMALIZACIÓN V2
Training normalizado: (147628, 20, 10)
Validation normalizado: (36907, 20, 10)
Held-out Test normalizado: (46134, 20, 10)

✅ MinMaxScaler ajustado exclusivamente con Training.
✅ Validation y Test fueron solo transformados.


## CONSTRUCCIÓN DEL MODELO HÍBRIDO CNN-LSTM

In [ ]:
# ============================================================
# MODELO CNN-LSTM - V2
# MISMA ARQUITECTURA DEL EXPERIMENTO ORIGINAL CORREGIDO
# ============================================================

import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Conv1D,
    MaxPooling1D,
    LSTM,
    Dropout,
    Dense
)

# ------------------------------------------------------------
# Reproducibilidad
# ------------------------------------------------------------

SEED = 42

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass


# ------------------------------------------------------------
# Dimensiones
# ------------------------------------------------------------

WINDOW_SIZE = 20
N_FEATURES = X_train.shape[2]
NUM_CLASSES = len(class_names)

print("WINDOW_SIZE:", WINDOW_SIZE)
print("N_FEATURES:", N_FEATURES)
print("NUM_CLASSES:", NUM_CLASSES)


# ============================================================
# ARQUITECTURA
# ============================================================

model_cnn_lstm = Sequential([

    Input(
        shape=(WINDOW_SIZE, N_FEATURES)
    ),

    # Convolutional feature extraction
    Conv1D(
        filters=64,
        kernel_size=3,
        activation="relu"
    ),

    MaxPooling1D(
        pool_size=2
    ),

    # Sequential modeling
    LSTM(
        units=32,
        return_sequences=False
    ),

    # Regularization
    Dropout(
        rate=0.30
    ),

    # Fully connected layer
    Dense(
        units=16,
        activation="relu"
    ),

    # Four-class classification
    Dense(
        units=NUM_CLASSES,
        activation="softmax"
    )
])


# ============================================================
# MODEL SUMMARY
# ============================================================

model_cnn_lstm.summary()

WINDOW_SIZE: 20
N_FEATURES: 10
NUM_CLASSES: 4


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 18, 64)         │         1,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 9, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │            68 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,996 (58.58 KB)

 Trainable params: 14,996 (58.58 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# ============================================================
# COMPILACIÓN CNN-LSTM - V2
# ============================================================

model_cnn_lstm.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("✅ CNN-LSTM V2 compilado correctamente.")

✅ CNN-LSTM V2 compilado correctamente.


In [ ]:
# ============================================================
# ENTRENAMIENTO CNN-LSTM - V2
# ============================================================

from time import perf_counter

EPOCHS = 15
BATCH_SIZE = 64

start_train = perf_counter()

history = model_cnn_lstm.fit(
    X_train,
    y_train,
    validation_data=(
        X_val,
        y_val
    ),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)

cnn_training_time = (
    perf_counter()
    - start_train
)

print("\n======================================")
print("CNN-LSTM V2 TRAINING TIME")
print("======================================")

print(
    f"Training Time: "
    f"{cnn_training_time:.4f} s"
)

Epoch 1/15
2307/2307 ━━━━━━━━━━━━━━━━━━━━ 30s 11ms/step - accuracy: 0.9149 - loss: 0.2028 - val_accuracy: 0.9806 - val_loss: 0.0507
Epoch 2/15
2307/2307 ━━━━━━━━━━━━━━━━━━━━ 27s 12ms/step - accuracy: 0.9890 - loss: 0.0320 - val_accuracy: 0.9888 - val_loss: 0.0303
Epoch 3/15
2307/2307 ━━━━━━━━━━━━━━━━━━━━ 28s 12ms/step - accuracy: 0.9956 - loss: 0.0140 - val_accuracy: 0.9958 - val_loss: 0.0120
Epoch 4/15
2307/2307 ━━━━━━━━━━━━━━━━━━━━ 26s 11ms/step - accuracy: 0.9970 - loss: 0.0089 - val_accuracy: 0.9988 - val_loss: 0.0037
Epoch 5/15
2307/2307 ━━━━━━━━━━━━━━━━━━━━ 25s 11ms/step - accuracy: 0.9979 - loss: 0.0068 - val_accuracy: 0.9988 - val_loss: 0.0034
Epoch 6/15
2307/2307 ━━━━━━━━━━━━━━━━━━━━ 44s 12ms/step - accuracy: 0.9986 - loss: 0.0046 - val_accuracy: 0.9988 - val_loss: 0.0041
Epoch 7/15
2307/2307 ━━━━━━━━━━━━━━━━━━━━ 27s 12ms/step - accuracy: 0.9988 - loss: 0.0037 - val_accuracy: 0.9992 - val_loss: 0.0024
Epoch 8/15
2307/2307 ━━━━━━━━━━━━━━━━━━━━ 40s 12ms/step - accuracy: 0.9990 -

In [ ]:
# ============================================================
# EVALUACIÓN FINAL CNN-LSTM V2
# HELD-OUT TEST = 46,134 SECUENCIAS
# ============================================================

import numpy as np
from time import perf_counter

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)


# ------------------------------------------------------------
# 1. Warm-up
# NO se incluye en el tiempo de inferencia
# ------------------------------------------------------------

_ = model_cnn_lstm.predict(
    X_test[:64],
    batch_size=64,
    verbose=0
)


# ------------------------------------------------------------
# 2. Inferencia sobre TODO el Held-out Test
# ------------------------------------------------------------

start_inference = perf_counter()

y_prob = model_cnn_lstm.predict(
    X_test,
    batch_size=64,
    verbose=0
)

cnn_inference_time = (
    perf_counter()
    - start_inference
)


# ------------------------------------------------------------
# 3. Probabilidades -> clases
# ------------------------------------------------------------

y_pred = np.argmax(
    y_prob,
    axis=1
)


# ------------------------------------------------------------
# 4. Métricas
# ------------------------------------------------------------

accuracy = accuracy_score(
    y_test,
    y_pred
)

macro_precision = precision_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

macro_recall = recall_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

macro_f1 = f1_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

weighted_f1 = f1_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0
)


# ------------------------------------------------------------
# 5. Correctos / errores
# ------------------------------------------------------------

errors = int(
    np.sum(y_test != y_pred)
)

correct = (
    len(y_test)
    - errors
)


# ------------------------------------------------------------
# 6. Rendimiento computacional
# ------------------------------------------------------------

latency_ms = (
    cnn_inference_time
    / len(X_test)
    * 1000
)

throughput = (
    len(X_test)
    / cnn_inference_time
)


# ============================================================
# RESULTADOS
# ============================================================

print("\n==============================================")
print("CNN-LSTM V2 — HELD-OUT TEST RESULTS")
print("==============================================")

print(
    f"Test sequences:     {len(y_test):,}"
)

print(
    f"Correct predictions:{correct:,}"
)

print(
    f"Misclassifications: {errors:,}"
)

print("\n----------------------------------------------")

print(
    f"Accuracy:        {accuracy * 100:.6f}%"
)

print(
    f"Macro Precision: {macro_precision * 100:.6f}%"
)

print(
    f"Macro Recall:    {macro_recall * 100:.6f}%"
)

print(
    f"Macro F1:        {macro_f1 * 100:.6f}%"
)

print(
    f"Weighted F1:     {weighted_f1 * 100:.6f}%"
)


print("\n==============================================")
print("COMPUTATIONAL PERFORMANCE")
print("==============================================")

print(
    f"Training Time:        {cnn_training_time:.4f} s"
)

print(
    f"Inference Time:       {cnn_inference_time:.4f} s"
)

print(
    f"Latency per sequence: {latency_ms:.6f} ms"
)

print(
    f"Throughput:           {throughput:.2f} sequences/s"
)


# ============================================================
# CLASSIFICATION REPORT
# ============================================================

print("\n==============================================")
print("CLASSIFICATION REPORT")
print("==============================================\n")

print(
    classification_report(
        y_test,
        y_pred,
        target_names=class_names,
        digits=6,
        zero_division=0
    )
)


# ============================================================
# CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    y_test,
    y_pred
)

print("\n==============================================")
print("CONFUSION MATRIX")
print("==============================================")

print(cm)


CNN-LSTM V2 — HELD-OUT TEST RESULTS
Test sequences:     46,134
Correct predictions:46,112
Misclassifications: 22

----------------------------------------------
Accuracy:        99.952313%
Macro Precision: 99.954710%
Macro Recall:    99.958962%
Macro F1:        99.956832%
Weighted F1:     99.952317%

COMPUTATIONAL PERFORMANCE
Training Time:        473.6181 s
Inference Time:       4.1975 s
Latency per sequence: 0.090986 ms
Throughput:           10990.72 sequences/s

CLASSIFICATION REPORT

                                  precision    recall  f1-score   support

         Attack_free_dataset.txt   0.999662  0.999451  0.999557     23694
          DoS_attack_dataset.txt   1.000000  0.999848  0.999924      6565
        Fuzzy_attack_dataset.txt   0.999831  0.999662  0.999747      5920
Impersonation_attack_dataset.txt   0.998695  0.999397  0.999046      9955

                        accuracy                       0.999523     46134
                       macro avg   0.999547  0.999590  0.999

In [ ]:
cnn_v2_y_pred = y_pred.copy()
cnn_v2_y_prob = y_prob.copy()
cnn_v2_y_test = y_test.copy()

cnn_v2_results = {
    "accuracy": accuracy,
    "macro_precision": macro_precision,
    "macro_recall": macro_recall,
    "macro_f1": macro_f1,
    "weighted_f1": weighted_f1,
    "errors": errors,
    "training_time": cnn_training_time,
    "inference_time": cnn_inference_time
}

print("✅ Resultados CNN-LSTM V2 preservados.")

✅ Resultados CNN-LSTM V2 preservados.


In [ ]:
# ============================================================
# PREPARACIÓN DE DATOS PARA MODELOS CONVENCIONALES - V2
# ============================================================

import numpy as np

# Las secuencias CNN-LSTM tienen forma:
# (samples, 20, 10)
#
# Para ML convencional:
# 20 x 10 = 200 características por secuencia

X_train_ml = X_train.reshape(
    X_train.shape[0],
    -1
).astype(np.float32)

X_val_ml = X_val.reshape(
    X_val.shape[0],
    -1
).astype(np.float32)

X_test_ml = X_test.reshape(
    X_test.shape[0],
    -1
).astype(np.float32)


print("==============================================")
print("CLASSICAL ML REPRESENTATION - V2")
print("==============================================")

print("Training:", X_train_ml.shape)
print("Validation:", X_val_ml.shape)
print("Held-out Test:", X_test_ml.shape)

# Verificaciones
assert X_train_ml.shape == (147628, 200)
assert X_val_ml.shape == (36907, 200)
assert X_test_ml.shape == (46134, 200)

assert len(y_train) == 147628
assert len(y_val) == 36907
assert len(y_test) == 46134

# Debe ser exactamente el mismo test que CNN-LSTM V2
assert np.array_equal(
    y_test,
    cnn_v2_y_test
)

print(
    "\n✅ Los modelos convencionales utilizarán "
    "exactamente el mismo Held-out Test que CNN-LSTM V2."
)

CLASSICAL ML REPRESENTATION - V2
Training: (147628, 200)
Validation: (36907, 200)
Held-out Test: (46134, 200)

✅ Los modelos convencionales utilizarán exactamente el mismo Held-out Test que CNN-LSTM V2.


In [ ]:
# ============================================================
# BASELINE MODELS - V2
# MISMO TRAINING Y MISMO HELD-OUT TEST DEL CNN-LSTM
# ============================================================

import numpy as np
import pandas as pd

from time import perf_counter

from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)


SEED = 42


# ============================================================
# MODELOS
# Mismas configuraciones utilizadas anteriormente
# ============================================================

models_v2 = {

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=SEED,
        n_jobs=-1
    ),

    "KNN": KNeighborsClassifier(
        n_neighbors=5,
        n_jobs=-1
    ),

    "Linear SVM": LinearSVC(
        C=1.0,
        random_state=SEED,
        max_iter=5000
    ),

    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=SEED
    ),

    "Naive Bayes": GaussianNB()
}


# ============================================================
# CONTENEDORES DE RESULTADOS
# ============================================================

results_v2 = []

predictions_v2 = {}

confusion_matrices_v2 = {}

classification_reports_v2 = {}


# ============================================================
# ENTRENAMIENTO Y EVALUACIÓN
# ============================================================

for model_name, model in models_v2.items():

    print("\n")
    print("=" * 60)
    print(model_name)
    print("=" * 60)

    # --------------------------------------------------------
    # TRAINING TIME
    # --------------------------------------------------------

    start_train = perf_counter()

    model.fit(
        X_train_ml,
        y_train
    )

    training_time = (
        perf_counter()
        - start_train
    )


    # --------------------------------------------------------
    # INFERENCE TIME
    # --------------------------------------------------------

    start_inference = perf_counter()

    y_pred_ml = model.predict(
        X_test_ml
    )

    inference_time = (
        perf_counter()
        - start_inference
    )


    # --------------------------------------------------------
    # MÉTRICAS
    # --------------------------------------------------------

    accuracy = accuracy_score(
        y_test,
        y_pred_ml
    )

    macro_precision = precision_score(
        y_test,
        y_pred_ml,
        average="macro",
        zero_division=0
    )

    macro_recall = recall_score(
        y_test,
        y_pred_ml,
        average="macro",
        zero_division=0
    )

    macro_f1 = f1_score(
        y_test,
        y_pred_ml,
        average="macro",
        zero_division=0
    )

    weighted_f1 = f1_score(
        y_test,
        y_pred_ml,
        average="weighted",
        zero_division=0
    )

    errors = int(
        np.sum(
            y_test != y_pred_ml
        )
    )

    correct = (
        len(y_test)
        - errors
    )


    # --------------------------------------------------------
    # LATENCIA Y THROUGHPUT
    # --------------------------------------------------------

    latency_ms = (
        inference_time
        / len(X_test_ml)
        * 1000
    )

    throughput = (
        len(X_test_ml)
        / inference_time
    )


    # --------------------------------------------------------
    # GUARDAR PREDICCIONES
    # Fundamental para McNemar posterior
    # --------------------------------------------------------

    predictions_v2[
        model_name
    ] = y_pred_ml.copy()


    # --------------------------------------------------------
    # MATRIZ DE CONFUSIÓN
    # --------------------------------------------------------

    cm = confusion_matrix(
        y_test,
        y_pred_ml
    )

    confusion_matrices_v2[
        model_name
    ] = cm


    # --------------------------------------------------------
    # CLASSIFICATION REPORT
    # --------------------------------------------------------

    report = classification_report(
        y_test,
        y_pred_ml,
        target_names=class_names,
        digits=6,
        zero_division=0,
        output_dict=True
    )

    classification_reports_v2[
        model_name
    ] = report


    # --------------------------------------------------------
    # GUARDAR RESULTADOS
    # --------------------------------------------------------

    results_v2.append({

        "Model": model_name,

        "Accuracy (%)":
            accuracy * 100,

        "Macro Precision (%)":
            macro_precision * 100,

        "Macro Recall (%)":
            macro_recall * 100,

        "Macro F1 (%)":
            macro_f1 * 100,

        "Weighted F1 (%)":
            weighted_f1 * 100,

        "Correct":
            correct,

        "Errors":
            errors,

        "Training Time (s)":
            training_time,

        "Inference Time (s)":
            inference_time,

        "Latency (ms/sequence)":
            latency_ms,

        "Throughput (sequences/s)":
            throughput
    })


    # --------------------------------------------------------
    # MOSTRAR RESULTADO
    # --------------------------------------------------------

    print(
        f"Accuracy:        "
        f"{accuracy * 100:.6f}%"
    )

    print(
        f"Macro Precision: "
        f"{macro_precision * 100:.6f}%"
    )

    print(
        f"Macro Recall:    "
        f"{macro_recall * 100:.6f}%"
    )

    print(
        f"Macro F1:        "
        f"{macro_f1 * 100:.6f}%"
    )

    print(
        f"Weighted F1:     "
        f"{weighted_f1 * 100:.6f}%"
    )

    print(
        f"Errors:          "
        f"{errors:,}"
    )

    print(
        f"Training Time:   "
        f"{training_time:.4f} s"
    )

    print(
        f"Inference Time:  "
        f"{inference_time:.4f} s"
    )


# ============================================================
# TABLA FINAL DE LOS CINCO BASELINES
# ============================================================

results_v2_df = pd.DataFrame(
    results_v2
)

results_v2_df = (
    results_v2_df
    .sort_values(
        by="Accuracy (%)",
        ascending=False
    )
    .reset_index(drop=True)
)


print("\n\n")
print("=" * 100)
print("BASELINE COMPARISON — V2")
print("=" * 100)

print(
    results_v2_df[
        [
            "Model",
            "Accuracy (%)",
            "Macro Precision (%)",
            "Macro Recall (%)",
            "Macro F1 (%)",
            "Weighted F1 (%)",
            "Errors",
            "Training Time (s)",
            "Inference Time (s)"
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print(
    "\n✅ Los cinco modelos convencionales V2 "
    "fueron evaluados sobre las mismas "
    "46,134 secuencias del Held-out Test."
)



Random Forest
Accuracy:        99.978324%
Macro Precision: 99.986284%
Macro Recall:    99.970167%
Macro F1:        99.978221%
Weighted F1:     99.978323%
Errors:          10
Training Time:   144.1717 s
Inference Time:  1.4758 s


KNN
Accuracy:        81.321802%
Macro Precision: 85.820734%
Macro Recall:    79.399276%
Macro F1:        81.432103%
Weighted F1:     81.325019%
Errors:          8,617
Training Time:   0.0295 s
Inference Time:  132.0394 s


Linear SVM
Accuracy:        88.522565%
Macro Precision: 90.467359%
Macro Recall:    90.187722%
Macro F1:        90.323560%
Weighted F1:     88.562128%
Errors:          5,295
Training Time:   42.8391 s
Inference Time:  0.0863 s


Logistic Regression
Accuracy:        88.865045%
Macro Precision: 90.681081%
Macro Recall:    90.740117%
Macro F1:        90.702823%
Weighted F1:     88.928492%
Errors:          5,137
Training Time:   102.7668 s
Inference Time:  0.1548 s


Naive Bayes
Accuracy:        88.015347%
Macro Precision: 90.390774%
Macro Rec

In [ ]:
# ============================================================
# AÑADIR CNN-LSTM V2 A LA COMPARACIÓN FINAL
# ============================================================

cnn_row = pd.DataFrame([{

    "Model": "CNN-LSTM",

    "Accuracy (%)":
        cnn_v2_results["accuracy"] * 100,

    "Macro Precision (%)":
        cnn_v2_results["macro_precision"] * 100,

    "Macro Recall (%)":
        cnn_v2_results["macro_recall"] * 100,

    "Macro F1 (%)":
        cnn_v2_results["macro_f1"] * 100,

    "Weighted F1 (%)":
        cnn_v2_results["weighted_f1"] * 100,

    "Correct":
        len(cnn_v2_y_test)
        - cnn_v2_results["errors"],

    "Errors":
        cnn_v2_results["errors"],

    "Training Time (s)":
        cnn_v2_results["training_time"],

    "Inference Time (s)":
        cnn_v2_results["inference_time"]
}])


final_comparison_v2 = pd.concat(
    [
        cnn_row,
        results_v2_df
    ],
    ignore_index=True
)


final_comparison_v2 = (
    final_comparison_v2
    .sort_values(
        by="Accuracy (%)",
        ascending=False
    )
    .reset_index(drop=True)
)


print("\n")
print("=" * 110)
print("FINAL MODEL COMPARISON — V2")
print("=" * 110)

print(
    final_comparison_v2[
        [
            "Model",
            "Accuracy (%)",
            "Macro Precision (%)",
            "Macro Recall (%)",
            "Macro F1 (%)",
            "Weighted F1 (%)",
            "Errors",
            "Training Time (s)",
            "Inference Time (s)"
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)



FINAL MODEL COMPARISON — V2
              Model  Accuracy (%)  Macro Precision (%)  Macro Recall (%)  Macro F1 (%)  Weighted F1 (%)  Errors  Training Time (s)  Inference Time (s)
      Random Forest     99.978324            99.986284         99.970167     99.978221        99.978323      10         144.171694            1.475776
           CNN-LSTM     99.952313            99.954710         99.958962     99.956832        99.952317      22         473.618125            4.197539
Logistic Regression     88.865045            90.681081         90.740117     90.702823        88.928492    5137         102.766812            0.154771
         Linear SVM     88.522565            90.467359         90.187722     90.323560        88.562128    5295          42.839087            0.086268
        Naive Bayes     88.015347            90.390774         88.944925     89.330974        88.357122    5529           0.356404            0.424033
                KNN     81.321802            85.820734         7

In [ ]:
# ============================================================
# EXACT McNEMAR TEST - V2
# CNN-LSTM vs RANDOM FOREST
# ============================================================

import numpy as np
from scipy.stats import binomtest

# ------------------------------------------------------------
# Predicciones definitivas
# ------------------------------------------------------------

cnn_pred = cnn_v2_y_pred

rf_pred = predictions_v2[
    "Random Forest"
]

# Verificar alineamiento
assert len(cnn_pred) == len(y_test)
assert len(rf_pred) == len(y_test)

assert np.array_equal(
    cnn_v2_y_test,
    y_test
)

# ------------------------------------------------------------
# Correcto / incorrecto por observación
# ------------------------------------------------------------

cnn_correct = (
    cnn_pred == y_test
)

rf_correct = (
    rf_pred == y_test
)

# ------------------------------------------------------------
# Tabla pareada McNemar
# ------------------------------------------------------------

both_correct = np.sum(
    cnn_correct & rf_correct
)

cnn_correct_rf_wrong = np.sum(
    cnn_correct & ~rf_correct
)

cnn_wrong_rf_correct = np.sum(
    ~cnn_correct & rf_correct
)

both_wrong = np.sum(
    ~cnn_correct & ~rf_correct
)

print("==============================================")
print("MCNEMAR CONTINGENCY TABLE — V2")
print("==============================================")

print(
    "Both correct:",
    f"{both_correct:,}"
)

print(
    "CNN correct / RF wrong:",
    f"{cnn_correct_rf_wrong:,}"
)

print(
    "CNN wrong / RF correct:",
    f"{cnn_wrong_rf_correct:,}"
)

print(
    "Both wrong:",
    f"{both_wrong:,}"
)

print(
    "\nTotal:",
    f"{both_correct + cnn_correct_rf_wrong + cnn_wrong_rf_correct + both_wrong:,}"
)

# ------------------------------------------------------------
# Exact McNemar test
#
# Solo se utilizan los pares discordantes
# ------------------------------------------------------------

b = int(
    cnn_correct_rf_wrong
)

c = int(
    cnn_wrong_rf_correct
)

discordant = b + c

if discordant > 0:

    result = binomtest(
        k=b,
        n=discordant,
        p=0.5,
        alternative="two-sided"
    )

    p_value = result.pvalue

else:

    p_value = 1.0


print("\n==============================================")
print("EXACT McNEMAR TEST — V2")
print("==============================================")

print(
    f"CNN correct / RF wrong (b): {b}"
)

print(
    f"CNN wrong / RF correct (c): {c}"
)

print(
    f"Discordant pairs (b+c):     {discordant}"
)

print(
    f"Exact p-value:               {p_value:.10f}"
)

# ------------------------------------------------------------
# Interpretación
# ------------------------------------------------------------

alpha = 0.05

print("\nInterpretation:")

if p_value < alpha:

    print(
        "Statistically significant difference "
        "(p < 0.05)."
    )

else:

    print(
        "No statistically significant difference "
        "(p >= 0.05)."
    )

MCNEMAR CONTINGENCY TABLE — V2
Both correct: 46,104
CNN correct / RF wrong: 8
CNN wrong / RF correct: 20
Both wrong: 2

Total: 46,134

EXACT McNEMAR TEST — V2
CNN correct / RF wrong (b): 8
CNN wrong / RF correct (c): 20
Discordant pairs (b+c):     28
Exact p-value:               0.0356981382

Interpretation:
Statistically significant difference (p < 0.05).


In [ ]:
# ============================================================
# FREEZE HELD-OUT TEST V2
# ============================================================

import numpy as np
import hashlib
import json
import os

FREEZE_DIR = "/content/drive/MyDrive/CAN_IDS_V2_FROZEN"
os.makedirs(FREEZE_DIR, exist_ok=True)


def array_sha256(array):

    array = np.ascontiguousarray(array)

    h = hashlib.sha256()

    h.update(str(array.shape).encode())
    h.update(str(array.dtype).encode())
    h.update(array.tobytes())

    return h.hexdigest()


# ------------------------------------------------------------
# Copias congeladas
# ------------------------------------------------------------

X_test_frozen = X_test.copy()
y_test_frozen = y_test.copy()


# ------------------------------------------------------------
# Fingerprints criptográficos
# ------------------------------------------------------------

X_test_hash = array_sha256(
    X_test_frozen
)

y_test_hash = array_sha256(
    y_test_frozen
)


print("X_test shape:", X_test_frozen.shape)
print("y_test shape:", y_test_frozen.shape)

print("\nSHA256 X_test:")
print(X_test_hash)

print("\nSHA256 y_test:")
print(y_test_hash)


# ------------------------------------------------------------
# Guardar físicamente
# ------------------------------------------------------------

np.savez_compressed(
    os.path.join(
        FREEZE_DIR,
        "heldout_test_v2.npz"
    ),
    X_test=X_test_frozen,
    y_test=y_test_frozen
)


metadata = {

    "dataset": "OTIDS V2 corrected",
    "seed": 42,

    "valid_messages": 4613437,
    "sequences": 230669,

    "training_sequences": 147628,
    "validation_sequences": 36907,
    "test_sequences": 46134,

    "window_size": 20,
    "features": 10,

    "X_test_sha256": X_test_hash,
    "y_test_sha256": y_test_hash
}


with open(
    os.path.join(
        FREEZE_DIR,
        "heldout_test_v2_metadata.json"
    ),
    "w"
) as f:

    json.dump(
        metadata,
        f,
        indent=4
    )


print(
    "\n✅ Held-out Test V2 congelado."
)

X_test shape: (46134, 20, 10)
y_test shape: (46134,)

SHA256 X_test:
c9cbfbdab16ffefe7c0790047a22980acb098113a2ce4eb242640a94449d1395

SHA256 y_test:
eadeb171940069c602f06efb5ddab6c4a8214db7ab5ec0c4116276ab1d873bce

✅ Held-out Test V2 congelado.


In [ ]:
# ============================================================
# ABLATION MODEL 1 — CNN ONLY
# ============================================================

import tensorflow as tf

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import (
    Input,
    Conv1D,
    MaxPooling1D,
    Flatten,
    Dropout,
    Dense
)


SEED = 42

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)


cnn_only = Sequential([

    Input(
        shape=(20, 10)
    ),

    Conv1D(
        filters=64,
        kernel_size=3,
        activation="relu"
    ),

    MaxPooling1D(
        pool_size=2
    ),

    Flatten(),

    Dropout(
        rate=0.30
    ),

    Dense(
        units=16,
        activation="relu"
    ),

    Dense(
        units=4,
        activation="softmax"
    )

])


cnn_only.compile(

    optimizer="adam",

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]

)


cnn_only.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 18, 64)         │         1,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 9, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 576)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 576)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │         9,232 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │            68 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,284 (44.08 KB)

 Trainable params: 11,284 (44.08 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# ============================================================
# ABLATION MODEL 2 — LSTM ONLY
# ============================================================

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)


lstm_only = Sequential([

    Input(
        shape=(20, 10)
    ),

    LSTM(
        units=32,
        return_sequences=False
    ),

    Dropout(
        rate=0.30
    ),

    Dense(
        units=16,
        activation="relu"
    ),

    Dense(
        units=4,
        activation="softmax"
    )

])


lstm_only.compile(

    optimizer="adam",

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]

)


lstm_only.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 32)             │         5,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │            68 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,100 (23.83 KB)

 Trainable params: 6,100 (23.83 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# ============================================================
# TRAIN ABLATION MODELS
# ============================================================

from time import perf_counter


EPOCHS = 15
BATCH_SIZE = 64


ablation_models = {

    "CNN-only": cnn_only,

    "LSTM-only": lstm_only

}


ablation_histories = {}
ablation_training_times = {}


for name, model in ablation_models.items():

    print("\n")
    print("=" * 60)
    print(name)
    print("=" * 60)

    start = perf_counter()

    history_ablation = model.fit(

        X_train,
        y_train,

        validation_data=(
            X_val,
            y_val
        ),

        epochs=EPOCHS,

        batch_size=BATCH_SIZE,

        verbose=1

    )

    elapsed = (
        perf_counter()
        - start
    )


    ablation_histories[
        name
    ] = history_ablation

    ablation_training_times[
        name
    ] = elapsed


    print(
        f"\n{name} Training Time: "
        f"{elapsed:.4f} s"
    )



CNN-only
Epoch 1/15
2307/2307 ━━━━━━━━━━━━━━━━━━━━ 21s 7ms/step - accuracy: 0.9056 - loss: 0.2321 - val_accuracy: 0.9794 - val_loss: 0.0623
Epoch 2/15
2307/2307 ━━━━━━━━━━━━━━━━━━━━ 16s 7ms/step - accuracy: 0.9802 - loss: 0.0556 - val_accuracy: 0.9921 - val_loss: 0.0267
Epoch 3/15
2307/2307 ━━━━━━━━━━━━━━━━━━━━ 16s 7ms/step - accuracy: 0.9886 - loss: 0.0322 - val_accuracy: 0.9948 - val_loss: 0.0172
Epoch 4/15
2307/2307 ━━━━━━━━━━━━━━━━━━━━ 17s 7ms/step - accuracy: 0.9929 - loss: 0.0212 - val_accuracy: 0.9979 - val_loss: 0.0087
Epoch 5/15
2307/2307 ━━━━━━━━━━━━━━━━━━━━ 23s 8ms/step - accuracy: 0.9948 - loss: 0.0158 - val_accuracy: 0.9980 - val_loss: 0.0086
Epoch 6/15
2307/2307 ━━━━━━━━━━━━━━━━━━━━ 18s 8ms/step - accuracy: 0.9960 - loss: 0.0112 - val_accuracy: 0.9989 - val_loss: 0.0048
Epoch 7/15
2307/2307 ━━━━━━━━━━━━━━━━━━━━ 18s 7ms/step - accuracy: 0.9973 - loss: 0.0086 - val_accuracy: 0.9993 - val_loss: 0.0029
Epoch 8/15
2307/2307 ━━━━━━━━━━━━━━━━━━━━ 16s 7ms/step - accuracy: 0.997

In [ ]:
# ============================================================
# FINAL ABLATION EVALUATION
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

from time import perf_counter


ablation_results = []

ablation_predictions = {}


for name, model in ablation_models.items():

    # Verificar que usamos el test congelado
    assert (
        array_sha256(X_test_frozen)
        == X_test_hash
    )

    assert (
        array_sha256(y_test_frozen)
        == y_test_hash
    )


    # Warm-up
    _ = model.predict(
        X_test_frozen[:64],
        verbose=0
    )


    # Inferencia
    start = perf_counter()

    probabilities = model.predict(

        X_test_frozen,

        batch_size=64,

        verbose=0
    )

    inference_time = (
        perf_counter()
        - start
    )


    predictions = np.argmax(
        probabilities,
        axis=1
    )


    ablation_predictions[
        name
    ] = predictions.copy()


    accuracy = accuracy_score(
        y_test_frozen,
        predictions
    )

    macro_precision = precision_score(
        y_test_frozen,
        predictions,
        average="macro",
        zero_division=0
    )

    macro_recall = recall_score(
        y_test_frozen,
        predictions,
        average="macro",
        zero_division=0
    )

    macro_f1 = f1_score(
        y_test_frozen,
        predictions,
        average="macro",
        zero_division=0
    )

    errors = int(
        np.sum(
            predictions
            != y_test_frozen
        )
    )


    ablation_results.append({

        "Model": name,

        "Parameters":
            model.count_params(),

        "Accuracy (%)":
            accuracy * 100,

        "Macro Precision (%)":
            macro_precision * 100,

        "Macro Recall (%)":
            macro_recall * 100,

        "Macro F1 (%)":
            macro_f1 * 100,

        "Errors":
            errors,

        "Training Time (s)":
            ablation_training_times[name],

        "Inference Time (s)":
            inference_time

    })


import pandas as pd

ablation_df = pd.DataFrame(
    ablation_results
)

print(
    ablation_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

    Model  Parameters  Accuracy (%)  Macro Precision (%)  Macro Recall (%)  Macro F1 (%)  Errors  Training Time (s)  Inference Time (s)
 CNN-only       11284     99.956648            99.969903         99.948126     99.959007      20         264.484407            2.936155
LSTM-only        6100     99.828760            99.915441         99.792167     99.853405      79         549.088555            3.419714


In [ ]:
cnn_lstm_ablation_row = pd.DataFrame([{

    "Model": "CNN-LSTM",

    "Parameters": 14996,

    "Accuracy (%)":
        cnn_v2_results["accuracy"] * 100,

    "Macro Precision (%)":
        cnn_v2_results["macro_precision"] * 100,

    "Macro Recall (%)":
        cnn_v2_results["macro_recall"] * 100,

    "Macro F1 (%)":
        cnn_v2_results["macro_f1"] * 100,

    "Errors":
        cnn_v2_results["errors"],

    "Training Time (s)":
        cnn_v2_results["training_time"],

    "Inference Time (s)":
        cnn_v2_results["inference_time"]

}])


ablation_final = pd.concat(

    [
        cnn_lstm_ablation_row,
        ablation_df
    ],

    ignore_index=True
)


print(
    "\n=============================================="
)

print(
    "ABLATION STUDY — FINAL"
)

print(
    "=============================================="
)


print(
    ablation_final.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


ABLATION STUDY — FINAL
    Model  Parameters  Accuracy (%)  Macro Precision (%)  Macro Recall (%)  Macro F1 (%)  Errors  Training Time (s)  Inference Time (s)
 CNN-LSTM       14996     99.952313            99.954710         99.958962     99.956832      22         473.618125            4.197539
 CNN-only       11284     99.956648            99.969903         99.948126     99.959007      20         264.484407            2.936155
LSTM-only        6100     99.828760            99.915441         99.792167     99.853405      79         549.088555            3.419714


In [ ]:
# ============================================================
# EXACT McNEMAR FOR ABLATION
# ============================================================

from scipy.stats import binomtest


def exact_mcnemar(
    y_true,
    pred_a,
    pred_b,
    name_a,
    name_b
):

    a_correct = (
        pred_a == y_true
    )

    b_correct = (
        pred_b == y_true
    )


    a_right_b_wrong = int(
        np.sum(
            a_correct
            & ~b_correct
        )
    )

    a_wrong_b_right = int(
        np.sum(
            ~a_correct
            & b_correct
        )
    )


    discordant = (
        a_right_b_wrong
        + a_wrong_b_right
    )


    if discordant == 0:

        p_value = 1.0

    else:

        p_value = binomtest(

            k=a_right_b_wrong,

            n=discordant,

            p=0.5,

            alternative="two-sided"

        ).pvalue


    print("\n")
    print("=" * 60)

    print(
        f"{name_a} vs {name_b}"
    )

    print("=" * 60)

    print(
        f"{name_a} correct / "
        f"{name_b} wrong: "
        f"{a_right_b_wrong}"
    )

    print(
        f"{name_a} wrong / "
        f"{name_b} correct: "
        f"{a_wrong_b_right}"
    )

    print(
        f"Exact McNemar p-value: "
        f"{p_value:.10f}"
    )


    return p_value

In [ ]:
p_cnn = exact_mcnemar(

    y_test_frozen,

    cnn_v2_y_pred,

    ablation_predictions["CNN-only"],

    "CNN-LSTM",

    "CNN-only"

)


p_lstm = exact_mcnemar(

    y_test_frozen,

    cnn_v2_y_pred,

    ablation_predictions["LSTM-only"],

    "CNN-LSTM",

    "LSTM-only"

)



CNN-LSTM vs CNN-only
CNN-LSTM correct / CNN-only wrong: 12
CNN-LSTM wrong / CNN-only correct: 14
Exact McNemar p-value: 0.8450189829


CNN-LSTM vs LSTM-only
CNN-LSTM correct / LSTM-only wrong: 73
CNN-LSTM wrong / LSTM-only correct: 16
Exact McNemar p-value: 0.0000000007


In [ ]:
cnn_weights_bytes = sum(

    weight.numpy().nbytes

    for weight
    in model_cnn_lstm.weights

)


cnn_weights_kib = (
    cnn_weights_bytes / 1024
)

cnn_weights_mib = (
    cnn_weights_bytes
    / (1024 ** 2)
)


print(
    "CNN-LSTM learned weights:"
)

print(
    f"{cnn_weights_bytes:,} bytes"
)

print(
    f"{cnn_weights_kib:.2f} KiB"
)

print(
    f"{cnn_weights_mib:.4f} MiB"
)

CNN-LSTM learned weights:
59,984 bytes
58.58 KiB
0.0572 MiB


In [ ]:
# ============================================================
# SERIALIZED MODEL SIZE
# ============================================================

import os
import joblib


MODEL_DIR = (
    "/content/drive/MyDrive/"
    "CAN_IDS_V2_MODEL_SIZE"
)

os.makedirs(
    MODEL_DIR,
    exist_ok=True
)


# ------------------------------------------------------------
# CNN-LSTM
# ------------------------------------------------------------

cnn_path = os.path.join(
    MODEL_DIR,
    "cnn_lstm_v2.keras"
)

model_cnn_lstm.save(
    cnn_path
)


cnn_file_size = (
    os.path.getsize(cnn_path)
)


# ------------------------------------------------------------
# RANDOM FOREST
# ------------------------------------------------------------

rf_model = models_v2[
    "Random Forest"
]

rf_path = os.path.join(
    MODEL_DIR,
    "random_forest_v2.joblib"
)

joblib.dump(
    rf_model,
    rf_path,
    compress=0
)


rf_file_size = (
    os.path.getsize(rf_path)
)


print(
    "CNN-LSTM serialized:"
)

print(
    f"{cnn_file_size / 1024:.2f} KiB"
)


print(
    "\nRandom Forest serialized:"
)

print(
    f"{rf_file_size / 1024:.2f} KiB"
)


print(
    "\nRF / CNN size ratio:"
)

print(
    f"{rf_file_size / cnn_file_size:.2f}x"
)

CNN-LSTM serialized:
213.67 KiB

Random Forest serialized:
145540.73 KiB

RF / CNN size ratio:
681.15x


In [ ]:
# ============================================================
# RANDOM FOREST COMPLEXITY
# ============================================================

rf_model = models_v2[
    "Random Forest"
]


tree_nodes = [

    estimator.tree_.node_count

    for estimator
    in rf_model.estimators_

]


total_nodes = sum(
    tree_nodes
)


mean_nodes = np.mean(
    tree_nodes
)


max_nodes = max(
    tree_nodes
)


print(
    "Number of trees:",
    len(rf_model.estimators_)
)

print(
    "Total tree nodes:",
    f"{total_nodes:,}"
)

print(
    "Mean nodes/tree:",
    f"{mean_nodes:.2f}"
)

print(
    "Largest tree:",
    f"{max_nodes:,}"
)

Number of trees: 100
Total tree nodes: 1,552,002
Mean nodes/tree: 15520.02
Largest tree: 18,387


In [ ]:
# ============================================================
# RANDOM FOREST CORE LEARNED MEMORY
# ============================================================

rf_core_bytes = 0


for estimator in rf_model.estimators_:

    tree = estimator.tree_

    arrays = [

        tree.children_left,
        tree.children_right,
        tree.feature,
        tree.threshold,
        tree.impurity,
        tree.n_node_samples,
        tree.weighted_n_node_samples,
        tree.value

    ]

    rf_core_bytes += sum(

        array.nbytes

        for array in arrays

    )


print(
    "Random Forest learned arrays:"
)

print(
    f"{rf_core_bytes:,} bytes"
)

print(
    f"{rf_core_bytes / 1024:.2f} KiB"
)

print(
    f"{rf_core_bytes / (1024**2):.4f} MiB"
)


print(
    "\nCNN-LSTM learned weights:"
)

print(
    f"{cnn_weights_bytes:,} bytes"
)

print(
    f"{cnn_weights_bytes / 1024:.2f} KiB"
)


print(
    "\nRF core arrays / CNN weights:"
)

print(
    f"{rf_core_bytes / cnn_weights_bytes:.2f}x"
)

Random Forest learned arrays:
136,576,176 bytes
133375.17 KiB
130.2492 MiB

CNN-LSTM learned weights:
59,984 bytes
58.58 KiB

RF core arrays / CNN weights:
2276.88x


In [ ]:
p_cnn = exact_mcnemar(
    y_test_frozen,
    cnn_v2_y_pred,
    ablation_predictions["CNN-only"],
    "CNN-LSTM",
    "CNN-only"
)

p_lstm = exact_mcnemar(
    y_test_frozen,
    cnn_v2_y_pred,
    ablation_predictions["LSTM-only"],
    "CNN-LSTM",
    "LSTM-only"
)



CNN-LSTM vs CNN-only
CNN-LSTM correct / CNN-only wrong: 12
CNN-LSTM wrong / CNN-only correct: 14
Exact McNemar p-value: 0.8450189829


CNN-LSTM vs LSTM-only
CNN-LSTM correct / LSTM-only wrong: 73
CNN-LSTM wrong / LSTM-only correct: 16
Exact McNemar p-value: 0.0000000007


**----------------------**

**MATRIZ DE CONFUSIÓN - CNN-LSTM**

In [ ]:
# ============================================================
# MATRIZ DE CONFUSIÓN DEFINITIVA - CNN-LSTM
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay
)

# Si todavía no existen las predicciones finales,
# volver a obtenerlas del modelo definitivo
if "y_pred" not in globals():

    y_prob = model_cnn_lstm.predict(
        X_test,
        batch_size=64,
        verbose=1
    )

    y_pred = np.argmax(
        y_prob,
        axis=1
    )


# Matriz de confusión
cm_cnn_lstm = confusion_matrix(
    y_test,
    y_pred
)

print("CONFUSION MATRIX - CNN-LSTM")
print(cm_cnn_lstm)


# Visualización
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_cnn_lstm,
    display_labels=label_encoder.classes_
)

fig, ax = plt.subplots(
    figsize=(9, 7)
)

disp.plot(
    ax=ax,
    values_format="d"
)

plt.xticks(
    rotation=30,
    ha="right"
)

plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")

plt.title(
    "Confusion Matrix of the Proposed CNN-LSTM Model"
)

plt.tight_layout()
plt.show()

**ROC-AUC MULTICLASE CNN-LSTM**

In [ ]:
# ============================================================
# ROC-AUC MULTICLASE - TEST INDEPENDIENTE
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc, roc_auc_score

# ------------------------------------------------------------
# Binarizar las etiquetas reales
# ------------------------------------------------------------

n_classes = len(label_encoder.classes_)

y_test_bin = label_binarize(
    y_test,
    classes=np.arange(n_classes)
)

# IMPORTANTE:
# y_prob debe provenir de:
# model_cnn_lstm.predict(X_test)
#
# NO utilizar y_pred para calcular ROC.
assert y_prob.shape == (len(y_test), n_classes)


# ------------------------------------------------------------
# ROC y AUC por clase
# ------------------------------------------------------------

fpr = {}
tpr = {}
roc_auc = {}

for i in range(n_classes):

    fpr[i], tpr[i], _ = roc_curve(
        y_test_bin[:, i],
        y_prob[:, i]
    )

    roc_auc[i] = auc(
        fpr[i],
        tpr[i]
    )


# ------------------------------------------------------------
# AUC macro y weighted
# ------------------------------------------------------------

macro_auc = roc_auc_score(
    y_test,
    y_prob,
    multi_class="ovr",
    average="macro"
)

weighted_auc = roc_auc_score(
    y_test,
    y_prob,
    multi_class="ovr",
    average="weighted"
)


print("ROC-AUC por clase:")

for i in range(n_classes):
    print(
        f"{label_encoder.classes_[i]}: "
        f"{roc_auc[i]:.6f}"
    )

print(
    f"\nMacro ROC-AUC: {macro_auc:.6f}"
)

print(
    f"Weighted ROC-AUC: {weighted_auc:.6f}"
)


# ============================================================
# FIGURA ROC
# ============================================================

plt.figure(figsize=(8, 6))

for i in range(n_classes):

    plt.plot(
        fpr[i],
        tpr[i],
        linewidth=2,
        label=(
            f"{label_encoder.classes_[i]} "
            f"(AUC = {roc_auc[i]:.4f})"
        )
    )

# Clasificador aleatorio
plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random classifier (AUC = 0.5000)"
)

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.01])

plt.xlabel("False Positive Rate (FPR)")
plt.ylabel("True Positive Rate (TPR)")

plt.title(
    "ROC Curves of the Proposed CNN-LSTM Model"
)

plt.legend(
    loc="lower right"
)

plt.grid(True)

plt.tight_layout()
plt.show()

**PRECISION-RECALL CURVES - CNN-LSTM**

In [ ]:
# ============================================================
# PRECISION-RECALL CURVES - CNN-LSTM
# INDEPENDENT TEST SET
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import label_binarize
from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score
)

# ------------------------------------------------------------
# Nombres limpios para la figura
# ------------------------------------------------------------

display_names = [
    "Attack-free",
    "DoS",
    "Fuzzy",
    "Impersonation"
]

n_classes = len(display_names)


# ------------------------------------------------------------
# Verificaciones
# ------------------------------------------------------------

assert len(y_test) == 44897

assert y_prob.shape == (
    len(y_test),
    n_classes
)

print("y_test:", y_test.shape)
print("y_prob:", y_prob.shape)


# ------------------------------------------------------------
# Binarización one-vs-rest
# ------------------------------------------------------------

y_test_bin = label_binarize(
    y_test,
    classes=np.arange(n_classes)
)


# ------------------------------------------------------------
# Precision-Recall y Average Precision por clase
# ------------------------------------------------------------

precision = {}
recall = {}
average_precision = {}

for i in range(n_classes):

    precision[i], recall[i], _ = precision_recall_curve(
        y_test_bin[:, i],
        y_prob[:, i]
    )

    average_precision[i] = average_precision_score(
        y_test_bin[:, i],
        y_prob[:, i]
    )


# ------------------------------------------------------------
# Macro y Weighted Average Precision
# ------------------------------------------------------------

macro_ap = average_precision_score(
    y_test_bin,
    y_prob,
    average="macro"
)

weighted_ap = average_precision_score(
    y_test_bin,
    y_prob,
    average="weighted"
)


# ------------------------------------------------------------
# Resultados numéricos
# ------------------------------------------------------------

print("\nAVERAGE PRECISION BY CLASS")

for i in range(n_classes):

    print(
        f"{display_names[i]}: "
        f"{average_precision[i]:.6f}"
    )

print(
    f"\nMacro Average Precision: "
    f"{macro_ap:.6f}"
)

print(
    f"Weighted Average Precision: "
    f"{weighted_ap:.6f}"
)


# ------------------------------------------------------------
# Gráfica
# ------------------------------------------------------------

plt.figure(figsize=(8, 6))

for i in range(n_classes):

    plt.plot(
        recall[i],
        precision[i],
        linewidth=2,
        label=(
            f"{display_names[i]} "
            f"(AP = {average_precision[i]:.4f})"
        )
    )

plt.xlim([0.0, 1.01])
plt.ylim([0.0, 1.01])

plt.xlabel("Recall")
plt.ylabel("Precision")

plt.title(
    "Precision-Recall Curves of the Proposed CNN-LSTM Model"
)

plt.legend(
    loc="lower left"
)

plt.grid(True)

plt.tight_layout()
plt.show()

**Comparar tu modelo CNN-LSTM con modelos**

In [ ]:
# ============================================================
# PREPARACIÓN DE DATOS PARA MODELOS CLÁSICOS
# MISMO TRAIN / VALIDATION / TEST DEL CNN-LSTM
# ============================================================

# Cada ventana 20 x 10 se convierte en un vector de 200 features

X_train_c = X_train.reshape(
    X_train.shape[0],
    -1
)

X_val_c = X_val.reshape(
    X_val.shape[0],
    -1
)

X_test_c = X_test.reshape(
    X_test.shape[0],
    -1
)

# Las etiquetas son EXACTAMENTE las mismas
y_train_c = y_train.copy()
y_val_c = y_val.copy()
y_test_c = y_test.copy()

print("Training clásico:", X_train_c.shape)
print("Validation clásico:", X_val_c.shape)
print("Test clásico:", X_test_c.shape)

print("y_train:", y_train_c.shape)
print("y_val:", y_val_c.shape)
print("y_test:", y_test_c.shape)

assert X_train_c.shape == (143668, 200)
assert X_val_c.shape == (35917, 200)
assert X_test_c.shape == (44897, 200)

assert len(y_test_c) == 44897

print("✅ Los modelos clásicos utilizarán exactamente el mismo split del CNN-LSTM.")

**-----------------**

In [ ]:
# ============================================================
# COMPARACIÓN CON MODELOS CLÁSICOS
# KNN - LOGISTIC REGRESSION - NAIVE BAYES
# RANDOM FOREST - SVM
# ============================================================

import time
import pandas as pd

from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)


# ============================================================
# MODELOS BASELINE
# ============================================================

models = {

    # --------------------------------------------------------
    # K-Nearest Neighbors
    # --------------------------------------------------------
    "KNN": KNeighborsClassifier(
        n_neighbors=5,
        n_jobs=-1
    ),

    # --------------------------------------------------------
    # Logistic Regression
    # --------------------------------------------------------
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=SEED
    ),

    # --------------------------------------------------------
    # Gaussian Naive Bayes
    # --------------------------------------------------------
    "Naive Bayes": GaussianNB(),

    # --------------------------------------------------------
    # Random Forest
    # --------------------------------------------------------
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=SEED,
        n_jobs=-1
    ),

    # --------------------------------------------------------
    # Support Vector Machine
    # LinearSVC se utiliza por la dimensión del dataset
    # --------------------------------------------------------
    "SVM": LinearSVC(
        C=1.0,
        random_state=SEED,
        max_iter=5000
    )
}


# ============================================================
# ALMACENAR RESULTADOS
# ============================================================

results = {}


# ============================================================
# ENTRENAMIENTO Y EVALUACIÓN
# ============================================================

for model_name, model in models.items():

    print("\n" + "=" * 70)
    print(f"MODELO: {model_name}")
    print("=" * 70)

    # --------------------------------------------------------
    # ENTRENAMIENTO
    # --------------------------------------------------------

    start_train = time.perf_counter()

    model.fit(
        X_train_c,
        y_train_c
    )

    training_time = (
        time.perf_counter()
        -
        start_train
    )


    # --------------------------------------------------------
    # PREDICCIÓN SOBRE EL MISMO TEST INDEPENDIENTE
    # --------------------------------------------------------

    start_test = time.perf_counter()

    y_pred_model = model.predict(
        X_test_c
    )

    inference_time = (
        time.perf_counter()
        -
        start_test
    )


    # --------------------------------------------------------
    # MÉTRICAS
    # --------------------------------------------------------

    accuracy = accuracy_score(
        y_test_c,
        y_pred_model
    )

    precision_macro = precision_score(
        y_test_c,
        y_pred_model,
        average="macro",
        zero_division=0
    )

    recall_macro = recall_score(
        y_test_c,
        y_pred_model,
        average="macro",
        zero_division=0
    )

    f1_macro = f1_score(
        y_test_c,
        y_pred_model,
        average="macro",
        zero_division=0
    )

    f1_weighted = f1_score(
        y_test_c,
        y_pred_model,
        average="weighted",
        zero_division=0
    )

    errors = int(
        (y_test_c != y_pred_model).sum()
    )


    # --------------------------------------------------------
    # GUARDAR RESULTADOS
    # --------------------------------------------------------

    results[model_name] = {

        "accuracy": accuracy,

        "macro_precision": precision_macro,

        "macro_recall": recall_macro,

        "macro_f1": f1_macro,

        "weighted_f1": f1_weighted,

        "errors": errors,

        "training_time": training_time,

        "inference_time": inference_time,

        "y_pred": y_pred_model,

        "report": classification_report(
            y_test_c,
            y_pred_model,
            target_names=label_encoder.classes_,
            output_dict=True,
            zero_division=0
        )
    }


    # --------------------------------------------------------
    # MOSTRAR RESULTADOS
    # --------------------------------------------------------

    print(
        f"Accuracy: "
        f"{accuracy * 100:.6f}%"
    )

    print(
        f"Macro Precision: "
        f"{precision_macro:.6f}"
    )

    print(
        f"Macro Recall: "
        f"{recall_macro:.6f}"
    )

    print(
        f"Macro F1: "
        f"{f1_macro:.6f}"
    )

    print(
        f"Weighted F1: "
        f"{f1_weighted:.6f}"
    )

    print(
        f"Misclassified sequences: "
        f"{errors:,}"
    )

    print(
        f"Training time: "
        f"{training_time:.3f} s"
    )

    print(
        f"Inference time: "
        f"{inference_time:.3f} s"
    )

    print("\nCLASSIFICATION REPORT\n")

    print(
        classification_report(
            y_test_c,
            y_pred_model,
            target_names=label_encoder.classes_,
            digits=6,
            zero_division=0
        )
    )

In [ ]:
# ============================================================
# AGREGAR CNN-LSTM A RESULTS
# CON TIEMPOS REALES
# ============================================================

cnn_accuracy = accuracy_score(
    y_test,
    y_pred
)

cnn_precision = precision_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

cnn_recall = recall_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

cnn_f1 = f1_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

cnn_weighted_f1 = f1_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0
)

cnn_errors = int(
    (y_test != y_pred).sum()
)


results["CNN-LSTM"] = {

    "accuracy": cnn_accuracy,

    "macro_precision": cnn_precision,

    "macro_recall": cnn_recall,

    "macro_f1": cnn_f1,

    "weighted_f1": cnn_weighted_f1,

    "errors": cnn_errors,

    # AQUÍ ESTABA EL PROBLEMA
    "training_time": cnn_training_time,

    "inference_time": cnn_inference_time,

    "y_pred": y_pred,

    "report": classification_report(
        y_test,
        y_pred,
        target_names=label_encoder.classes_,
        output_dict=True,
        zero_division=0
    )
}

print("✅ CNN-LSTM agregado con tiempos reales.")

print(
    "Training Time:",
    results["CNN-LSTM"]["training_time"]
)

print(
    "Inference Time:",
    results["CNN-LSTM"]["inference_time"]
)

In [ ]:
# ============================================================
# TABLA COMPARATIVA FINAL
# ============================================================

comparison_rows = []

for model_name, res in results.items():

    comparison_rows.append({

        "Model":
            model_name,

        "Accuracy (%)":
            res["accuracy"] * 100,

        "Macro Precision (%)":
            res["macro_precision"] * 100,

        "Macro Recall (%)":
            res["macro_recall"] * 100,

        "Macro F1 (%)":
            res["macro_f1"] * 100,

        "Weighted F1 (%)":
            res["weighted_f1"] * 100,

        "Errors":
            res["errors"],

        "Training Time (s)":
            res["training_time"],

        "Inference Time (s)":
            res["inference_time"]
    })


comparison_df = pd.DataFrame(
    comparison_rows
)

comparison_df = comparison_df.sort_values(
    by="Accuracy (%)",
    ascending=False
)

print("\n" + "=" * 100)
print("FINAL MODEL COMPARISON")
print("=" * 100)

print(
    comparison_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

In [ ]:
import numpy as np
from scipy.stats import binomtest

# Predicciones definitivas
pred_cnn = results["CNN-LSTM"]["y_pred"]
pred_rf = results["Random Forest"]["y_pred"]

true = y_test

cnn_correct = pred_cnn == true
rf_correct = pred_rf == true

# CNN acierta y RF falla
b = np.sum(cnn_correct & ~rf_correct)

# RF acierta y CNN falla
c = np.sum(~cnn_correct & rf_correct)

print("CNN correcto / RF incorrecto:", b)
print("CNN incorrecto / RF correcto:", c)

if b + c > 0:
    mcnemar_exact = binomtest(
        min(b, c),
        n=b + c,
        p=0.5,
        alternative="two-sided"
    )

    print("Exact McNemar p-value:", mcnemar_exact.pvalue)

**DATOS EXTRAS**

In [ ]:
import sys
import platform
import numpy as np
import pandas as pd
import sklearn
import tensorflow as tf

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Processor:", platform.processor())

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)
print("TensorFlow:", tf.__version__)
print("Keras:", tf.keras.__version__)

print("\nTensorFlow devices:")
print(tf.config.list_physical_devices())

In [ ]:
!lscpu | grep -E "Model name|CPU\\(s\\)|Thread|Core|Socket"

In [ ]:
# ============================================================
# AUDITORÍA DEL PARSER OTIDS
# Compara líneas físicas vs. registros reconocidos por el parser
# NO modifica df_all, df_clean ni el modelo
# ============================================================

import os
import re
import pandas as pd

folder = "/content/drive/MyDrive/Datasets"

files = [
    "Attack_free_dataset.txt",
    "DoS_attack_dataset.txt",
    "Fuzzy_attack_dataset.txt",
    "Impersonation_attack_dataset.txt"
]

# MISMO patrón utilizado en el pipeline definitivo
pattern = re.compile(
    r"Timestamp:\s+([\d.]+)\s+"
    r"ID:\s+([0-9A-Fa-f]+\s+[0-9]{3})\s+"
    r"DLC:\s+(\d+)\s+"
    r"(.*)"
)

audit_results = []

for file_name in files:

    file_path = os.path.join(folder, file_name)

    total_lines = 0
    matched_lines = 0
    unmatched_lines = 0
    examples_unmatched = []

    with open(
        file_path,
        "r",
        encoding="latin-1",
        errors="ignore"
    ) as f:

        for line in f:

            total_lines += 1

            line_clean = line.strip()

            match = pattern.search(line_clean)

            if match:
                matched_lines += 1
            else:
                unmatched_lines += 1

                # Guardar solo algunos ejemplos
                if len(examples_unmatched) < 5:
                    examples_unmatched.append(line_clean)

    audit_results.append({
        "file": file_name,
        "total_lines": total_lines,
        "matched_by_parser": matched_lines,
        "not_matched": unmatched_lines,
        "match_percentage": (
            matched_lines / total_lines * 100
            if total_lines > 0 else 0
        )
    })

    print("\n==============================================")
    print(file_name)
    print("==============================================")
    print(f"Líneas físicas totales: {total_lines:,}")
    print(f"Registros reconocidos:  {matched_lines:,}")
    print(f"No reconocidos:         {unmatched_lines:,}")
    print(
        f"Porcentaje reconocido: "
        f"{matched_lines / total_lines * 100:.6f}%"
    )

    if examples_unmatched:
        print("\nEjemplos de líneas NO reconocidas:")
        for i, example in enumerate(examples_unmatched, 1):
            print(f"{i}: {repr(example)}")


audit_df = pd.DataFrame(audit_results)

print("\n\n==============================================")
print("RESUMEN GLOBAL")
print("==============================================")
print(audit_df.to_string(index=False))

print("\nTOTALES:")
print(
    "Líneas físicas:",
    f"{audit_df['total_lines'].sum():,}"
)
print(
    "Reconocidas por parser:",
    f"{audit_df['matched_by_parser'].sum():,}"
)
print(
    "No reconocidas:",
    f"{audit_df['not_matched'].sum():,}"
)

In [ ]:
# ============================================================
# AUDITORÍA DETALLADA DE LÍNEAS NO RECONOCIDAS - OTIDS
# ============================================================
# IMPORTANTE:
# - NO modifica df_all
# - NO modifica df_clean
# - NO modifica X, y ni los modelos
# - Solo analiza las líneas descartadas por el parser original
# ============================================================

import os
import re
import pandas as pd
from collections import Counter, defaultdict

folder = "/content/drive/MyDrive/Datasets"

files = [
    "Attack_free_dataset.txt",
    "DoS_attack_dataset.txt",
    "Fuzzy_attack_dataset.txt",
    "Impersonation_attack_dataset.txt"
]

# ============================================================
# 1. PATRÓN ORIGINAL DEL EXPERIMENTO
# ============================================================

original_pattern = re.compile(
    r"Timestamp:\s+([\d.]+)\s+"
    r"ID:\s+([0-9A-Fa-f]+\s+[0-9]{3})\s+"
    r"DLC:\s+(\d+)\s+"
    r"(.*)"
)

# ============================================================
# 2. PATRONES AUXILIARES PARA DIAGNÓSTICO
# ============================================================

timestamp_tag = re.compile(r"Timestamp:", re.IGNORECASE)
id_tag = re.compile(r"\bID:", re.IGNORECASE)
dlc_tag = re.compile(r"\bDLC:", re.IGNORECASE)

# Extraer contenido entre campos de forma más flexible
timestamp_value_pattern = re.compile(
    r"Timestamp:\s*(.*?)\s+ID:",
    re.IGNORECASE
)

id_value_pattern = re.compile(
    r"ID:\s*(.*?)\s+DLC:",
    re.IGNORECASE
)

dlc_and_rest_pattern = re.compile(
    r"DLC:\s*(\S+)(?:\s+(.*))?$",
    re.IGNORECASE
)

# ID exigido por el parser original
strict_id_pattern = re.compile(
    r"^[0-9A-Fa-f]+\s+[0-9]{3}$"
)

# Posible ID hexadecimal único
single_hex_id_pattern = re.compile(
    r"^[0-9A-Fa-f]+$"
)

# Timestamp esperado
valid_timestamp_pattern = re.compile(
    r"^\d+(?:\.\d+)?$"
)

# DLC numérico
valid_dlc_pattern = re.compile(
    r"^\d+$"
)


# ============================================================
# 3. FUNCIÓN PARA CLASIFICAR CADA LÍNEA NO RECONOCIDA
# ============================================================

def classify_unmatched_line(line):

    raw = line.rstrip("\n\r")
    clean = raw.strip()

    # A. Línea vacía
    if clean == "":
        return "EMPTY_LINE"

    # B. Faltan etiquetas principales
    if not timestamp_tag.search(clean):
        return "MISSING_TIMESTAMP_TAG"

    if not id_tag.search(clean):
        return "MISSING_ID_TAG"

    if not dlc_tag.search(clean):
        return "MISSING_DLC_TAG"

    # C. Revisar Timestamp
    ts_match = timestamp_value_pattern.search(clean)

    if ts_match is None:
        return "TIMESTAMP_STRUCTURE_MISMATCH"

    ts_value = ts_match.group(1).strip()

    if not valid_timestamp_pattern.fullmatch(ts_value):
        return "INVALID_TIMESTAMP_FORMAT"

    # D. Revisar ID
    id_match = id_value_pattern.search(clean)

    if id_match is None:
        return "ID_STRUCTURE_MISMATCH"

    id_value = id_match.group(1).strip()

    # Si tiene un único token hexadecimal
    if single_hex_id_pattern.fullmatch(id_value):
        return "SINGLE_TOKEN_CAN_ID"

    # Si NO cumple <HEX> <3 dígitos>
    if not strict_id_pattern.fullmatch(id_value):
        return "NONSTANDARD_CAN_ID_FORMAT"

    # E. Revisar DLC y payload
    dlc_match = dlc_and_rest_pattern.search(clean)

    if dlc_match is None:
        return "DLC_STRUCTURE_MISMATCH"

    dlc_value = dlc_match.group(1).strip()
    payload = dlc_match.group(2)

    if not valid_dlc_pattern.fullmatch(dlc_value):
        return "INVALID_DLC_FORMAT"

    # El patrón original requiere contenido/espacio después de DLC
    if payload is None or payload.strip() == "":
        return "EMPTY_OR_MISSING_PAYLOAD"

    # F. Si aparentemente todo está bien pero no coincidió
    return "OTHER_FORMAT_MISMATCH"


# ============================================================
# 4. RECORRER ARCHIVOS
# ============================================================

summary_rows = []
detail_rows = []

examples = defaultdict(list)

global_counter = Counter()

for file_name in files:

    file_path = os.path.join(folder, file_name)

    file_counter = Counter()

    total_lines = 0
    matched_lines = 0
    unmatched_lines = 0

    with open(
        file_path,
        "r",
        encoding="latin-1",
        errors="ignore"
    ) as f:

        for line_number, line in enumerate(f, start=1):

            total_lines += 1

            clean = line.strip()

            # Si el parser original la reconoce, no se audita
            if original_pattern.search(clean):
                matched_lines += 1
                continue

            unmatched_lines += 1

            category = classify_unmatched_line(line)

            file_counter[category] += 1
            global_counter[category] += 1

            # Guardar hasta 5 ejemplos por archivo/categoría
            key = (file_name, category)

            if len(examples[key]) < 5:
                examples[key].append({
                    "line_number": line_number,
                    "content": clean[:500]
                })

            detail_rows.append({
                "file": file_name,
                "line_number": line_number,
                "category": category,
                "content": clean[:1000]
            })

    for category, count in file_counter.items():

        summary_rows.append({
            "file": file_name,
            "category": category,
            "count": count,
            "percentage_of_unmatched": (
                count / unmatched_lines * 100
                if unmatched_lines > 0 else 0
            )
        })

    print("\n" + "=" * 70)
    print(file_name)
    print("=" * 70)

    print(f"Líneas totales:       {total_lines:,}")
    print(f"Reconocidas:          {matched_lines:,}")
    print(f"No reconocidas:       {unmatched_lines:,}")

    print("\nClasificación:")

    for category, count in file_counter.most_common():
        percentage = (
            count / unmatched_lines * 100
            if unmatched_lines > 0 else 0
        )

        print(
            f"{category:35s} "
            f"{count:10,} "
            f"({percentage:8.4f}%)"
        )


# ============================================================
# 5. RESUMEN GLOBAL
# ============================================================

print("\n\n" + "=" * 70)
print("RESUMEN GLOBAL DE LAS LÍNEAS NO RECONOCIDAS")
print("=" * 70)

total_unmatched = sum(global_counter.values())

for category, count in global_counter.most_common():

    percentage = (
        count / total_unmatched * 100
        if total_unmatched > 0 else 0
    )

    print(
        f"{category:35s} "
        f"{count:10,} "
        f"({percentage:8.4f}%)"
    )

print("\nTOTAL NO RECONOCIDAS:", f"{total_unmatched:,}")


# ============================================================
# 6. CREAR TABLAS
# ============================================================

audit_summary_df = pd.DataFrame(summary_rows)

audit_detail_df = pd.DataFrame(detail_rows)

print("\n\nTABLA RESUMEN:")
print(
    audit_summary_df
    .sort_values(
        ["file", "count"],
        ascending=[True, False]
    )
    .to_string(index=False)
)


# ============================================================
# 7. MOSTRAR EJEMPLOS
# ============================================================

print("\n\n" + "=" * 70)
print("EJEMPLOS POR CATEGORÍA")
print("=" * 70)

for key, records in examples.items():

    file_name, category = key

    print("\n")
    print("-" * 70)
    print("Archivo:", file_name)
    print("Categoría:", category)
    print("-" * 70)

    for record in records:
        print(
            f"Línea {record['line_number']}: "
            f"{repr(record['content'])}"
        )


# ============================================================
# 8. GUARDAR RESULTADOS
# ============================================================

summary_path = "/content/otids_unmatched_summary.csv"
detail_path = "/content/otids_unmatched_detail.csv"

audit_summary_df.to_csv(
    summary_path,
    index=False
)

audit_detail_df.to_csv(
    detail_path,
    index=False
)

print("\n\n✅ Auditoría terminada.")
print("Resumen guardado en:", summary_path)
print("Detalle guardado en:", detail_path)

----------------

In [ ]:
# ============================================================
# COMPROBAR DATOS V2 ANTES DE GUARDAR
# ============================================================

variables = [
    "X_train",
    "y_train",
    "X_val",
    "y_val"
]

for v in variables:
    print(
        v,
        "✅ existe" if v in globals()
        else "❌ NO existe"
    )

X_train ✅ existe
y_train ✅ existe
X_val ✅ existe
y_val ✅ existe


In [ ]:
# ============================================================
# EXPORTAR DEVELOPMENT SET V2 PARA CNN-LSTM V3
# ============================================================

import os
import numpy as np
import hashlib
import json

SAVE_DIR = "/content/drive/MyDrive/CAN_IDS_V3"

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)

FILE_PATH = os.path.join(
    SAVE_DIR,
    "development_v2_train_validation.npz"
)


# ------------------------------------------------------------
# Verificaciones antes de guardar
# ------------------------------------------------------------

assert X_train.shape == (147628, 20, 10)
assert y_train.shape == (147628,)

assert X_val.shape == (36907, 20, 10)
assert y_val.shape == (36907,)


print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_val:", X_val.shape)
print("y_val:", y_val.shape)


# ------------------------------------------------------------
# Guardar
# ------------------------------------------------------------

print("\nGuardando archivo...")

np.savez_compressed(
    FILE_PATH,
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val
)


# ------------------------------------------------------------
# Comprobar creación
# ------------------------------------------------------------

print("\n==============================================")
print("EXPORTACIÓN COMPLETADA")
print("==============================================")

print("Archivo:")
print(FILE_PATH)

print(
    "\n¿Existe?:",
    os.path.isfile(FILE_PATH)
)

print(
    "Tamaño:",
    f"{os.path.getsize(FILE_PATH)/(1024**2):.2f} MiB"
)


assert os.path.isfile(FILE_PATH)

print(
    "\n✅ Development dataset V2 guardado para V3."
)

X_train: (147628, 20, 10)
y_train: (147628,)
X_val: (36907, 20, 10)
y_val: (36907,)

Guardando archivo...

EXPORTACIÓN COMPLETADA
Archivo:
/content/drive/MyDrive/CAN_IDS_V3/development_v2_train_validation.npz

¿Existe?: True
Tamaño: 20.19 MiB

✅ Development dataset V2 guardado para V3.
